# Feature Engineering

In [4]:
# Load canonical dataset

import pandas as pd
import numpy as np

DATA_PATH = "../data/processed/ronaldo_1000_ml_canonical_final.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique match IDs:", df["match_id"].nunique())
print("Total goals:", df["target_goals"].sum())

print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

print("Chronological:", df["date"].is_monotonic_increasing)

Shape: (1333, 92)
Rows: 1333
Columns: 92
Unique match IDs: 1333
Total goals: 978
Date range: 2002-08-14 00:00:00 to 2026-08-28 00:00:00
Chronological: True


## Classify all columns

In [8]:
# Classify dataset columns

target_columns = [
    "target_goals",
    "target_scored",
    "target_multi_goal",
]

identifier_columns = [
    "match_id",
    "date",
]

postmatch_reference_columns = [
    "team_goals",
    "opponent_goals",
    "result",
    "ronaldo_goals",
    "ronaldo_assists",
    "source_status",
]

categorical_features = [
    "season",
    "team",
    "opponent",
    "competition",
    "competition_type",
    "venue",
    "day_of_week",
]

excluded_columns = (
    target_columns
    + identifier_columns
    + postmatch_reference_columns
)

model_feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

numeric_features = [
    col for col in model_feature_columns
    if col not in categorical_features
]

print("Targets:", len(target_columns))
print("Identifiers:", len(identifier_columns))
print("Post-match reference/leakage:", len(postmatch_reference_columns))
print("Categorical predictors:", len(categorical_features))
print("Numeric predictors:", len(numeric_features))
print("Total model predictors:", len(model_feature_columns))

print("\nTotal classified columns:")
print(
    len(target_columns)
    + len(identifier_columns)
    + len(postmatch_reference_columns)
    + len(model_feature_columns)
)

print("\nUnclassified columns:")
classified = set(
    target_columns
    + identifier_columns
    + postmatch_reference_columns
    + model_feature_columns
)

unclassified = [
    col for col in df.columns
    if col not in classified
]

print(unclassified)

print("\nCategorical predictors:")
print(categorical_features)

Targets: 3
Identifiers: 2
Post-match reference/leakage: 6
Categorical predictors: 7
Numeric predictors: 74
Total model predictors: 81

Total classified columns:
92

Unclassified columns:
[]

Categorical predictors:
['season', 'team', 'opponent', 'competition', 'competition_type', 'venue', 'day_of_week']


## Audit feature data types

In [11]:
# Validate predictor data types

numeric_type_check = df[numeric_features].dtypes.value_counts()

print("Numeric predictor dtypes:")
print(numeric_type_check)

print("\nNon-numeric columns accidentally inside numeric_features:")

bad_numeric = [
    col for col in numeric_features
    if not pd.api.types.is_numeric_dtype(df[col])
]

print(bad_numeric)

print("\nCategorical predictor dtypes:")

for col in categorical_features:
    print(
        f"{col:<25} "
        f"dtype={str(df[col].dtype):<10} "
        f"unique={df[col].nunique():>3} "
        f"missing={df[col].isna().sum():>3}"
    )

Numeric predictor dtypes:
int64      42
float64    32
Name: count, dtype: int64

Non-numeric columns accidentally inside numeric_features:
[]

Categorical predictor dtypes:
season                    dtype=object     unique= 25 missing=  0
team                      dtype=object     unique=  6 missing=  0
opponent                  dtype=object     unique=283 missing=  0
competition               dtype=object     unique= 31 missing=  0
competition_type          dtype=object     unique=  4 missing=  0
venue                     dtype=object     unique=  3 missing=  0
day_of_week               dtype=object     unique=  7 missing=  0


## Identify exact and near-duplicate numeric features

In [14]:
# Find exact and near-duplicate numeric predictors

corr_matrix = df[numeric_features].corr()

near_duplicate_pairs = []

for i in range(len(numeric_features)):
    for j in range(i + 1, len(numeric_features)):
        feature_a = numeric_features[i]
        feature_b = numeric_features[j]

        corr_value = corr_matrix.loc[feature_a, feature_b]

        if pd.notna(corr_value) and abs(corr_value) >= 0.98:
            near_duplicate_pairs.append(
                {
                    "feature_a": feature_a,
                    "feature_b": feature_b,
                    "correlation": corr_value,
                }
            )

near_duplicate_df = pd.DataFrame(near_duplicate_pairs)

if not near_duplicate_df.empty:
    near_duplicate_df["abs_correlation"] = (
        near_duplicate_df["correlation"].abs()
    )

    near_duplicate_df = (
        near_duplicate_df
        .sort_values("abs_correlation", ascending=False)
        .drop(columns="abs_correlation")
    )

    print(near_duplicate_df.to_string(index=False))
else:
    print("No predictor pairs found with |correlation| >= 0.98")

print("\nNumber of near-duplicate pairs:", len(near_duplicate_df))

             feature_a              feature_b  correlation
      career_goals_pre      goals_to_1000_pre    -1.000000
          goals_last_5  goals_per_game_last_5     1.000000
             age_years               age_days     1.000000
         goals_last_10 goals_per_game_last_10     0.999792
             age_years career_appearances_pre     0.999524
              age_days career_appearances_pre     0.999524
         goals_last_20 goals_per_game_last_20     0.998612
career_appearances_pre     career_assists_pre     0.995126
              age_days       career_goals_pre     0.993404
              age_days      goals_to_1000_pre    -0.993404
             age_years       career_goals_pre     0.993404
             age_years      goals_to_1000_pre    -0.993404
career_appearances_pre       career_goals_pre     0.993392
career_appearances_pre      goals_to_1000_pre    -0.993392
      career_goals_pre     career_assists_pre     0.993174
    career_assists_pre      goals_to_1000_pre    -0.9931

In [16]:
# Verify deterministic feature relationships

checks = {
    "goals_to_1000 = 1000 - career_goals": np.allclose(
        df["goals_to_1000_pre"],
        1000 - df["career_goals_pre"]
    ),

    "last_5_gpg = last_5_goals / available_matches": np.allclose(
        df["goals_per_game_last_5"],
        df["goals_last_5"] /
        np.minimum(df["career_appearances_pre"], 5).replace(0, np.nan),
        equal_nan=True
    ),

    "age_years approximately age_days / 365.25": np.allclose(
        df["age_years"],
        df["age_days"] / 365.25,
        atol=0.01
    ),
}

for name, result in checks.items():
    print(f"{name:<60} {result}")

print("\nMaximum differences:")

print(
    "goals_to_1000:",
    (
        df["goals_to_1000_pre"]
        - (1000 - df["career_goals_pre"])
    ).abs().max()
)

print(
    "age:",
    (
        df["age_years"]
        - df["age_days"] / 365.25
    ).abs().max()
)

goals_to_1000 = 1000 - career_goals                          True
last_5_gpg = last_5_goals / available_matches                False
age_years approximately age_days / 365.25                    True

Maximum differences:
goals_to_1000: 0
age: 0.0008914442162861747


In [18]:
# Inspect rolling goal-rate relationships

rolling_pairs = [
    ("goals_last_5", "goals_per_game_last_5", 5),
    ("goals_last_10", "goals_per_game_last_10", 10),
    ("goals_last_20", "goals_per_game_last_20", 20),
]

for goals_col, rate_col, window in rolling_pairs:
    available_matches = np.minimum(
        df["career_appearances_pre"],
        window
    )

    expected_rate = (
        df[goals_col]
        / pd.Series(available_matches, index=df.index).replace(0, np.nan)
    )

    difference = (df[rate_col] - expected_rate).abs()

    print(f"\n{goals_col} vs {rate_col}")
    print("Max difference:", difference.max())
    print("Rows different:", (difference > 1e-9).sum())

    mismatches = df.loc[
        difference > 1e-9,
        [
            "date",
            "career_appearances_pre",
            goals_col,
            rate_col,
        ]
    ].copy()

    if not mismatches.empty:
        mismatches["expected_rate"] = expected_rate.loc[mismatches.index]

        print(mismatches.head(10).to_string(index=False))


goals_last_5 vs goals_per_game_last_5
Max difference: 0.0
Rows different: 0

goals_last_10 vs goals_per_game_last_10
Max difference: 5.551115123125783e-17
Rows different: 0

goals_last_20 vs goals_per_game_last_20
Max difference: 5.551115123125783e-17
Rows different: 0


In [20]:
# Remove only deterministic/redundant predictors

deterministic_exclusions = [
    "age_days",
    "goals_last_5",
    "goals_last_10",
    "goals_last_20",
    "goals_to_1000_pre",
]

engineered_feature_columns = [
    col for col in model_feature_columns
    if col not in deterministic_exclusions
]

engineered_numeric_features = [
    col for col in numeric_features
    if col not in deterministic_exclusions
]

engineered_categorical_features = categorical_features.copy()

print("Original predictors:", len(model_feature_columns))
print("Excluded deterministic predictors:", len(deterministic_exclusions))
print("Remaining predictors:", len(engineered_feature_columns))

print("\nExcluded:")
for col in deterministic_exclusions:
    print(" -", col)

print("\nRemaining numeric:", len(engineered_numeric_features))
print("Remaining categorical:", len(engineered_categorical_features))

print(
    "\nFeature count check:",
    len(engineered_numeric_features)
    + len(engineered_categorical_features)
    == len(engineered_feature_columns)
)

Original predictors: 81
Excluded deterministic predictors: 5
Remaining predictors: 76

Excluded:
 - age_days
 - goals_last_5
 - goals_last_10
 - goals_last_20
 - goals_to_1000_pre

Remaining numeric: 69
Remaining categorical: 7

Feature count check: True


## Examine the remaining career-progression feature

In [24]:
# Inspect career-progression features

career_features = [
    "age_years",
    "career_appearances_pre",
    "career_goals_pre",
    "career_assists_pre",
    "career_goals_per_game_pre",
]

career_corr = df[career_features + ["target_goals"]].corr()

print("Career feature correlations:")
print(
    career_corr
    .round(3)
    .to_string()
)

print("\nFeature ranges:")

for col in career_features:
    print(
        f"{col:<30} "
        f"min={df[col].min():>10.3f}  "
        f"median={df[col].median():>10.3f}  "
        f"max={df[col].max():>10.3f}"
    )

print("\nLatest match values:")

print(
    df[
        ["date"] + career_features + ["target_goals"]
    ]
    .tail(1)
    .to_string(index=False)
)

Career feature correlations:
                           age_years  career_appearances_pre  career_goals_pre  career_assists_pre  career_goals_per_game_pre  target_goals
age_years                      1.000                   1.000             0.993               0.993                      0.935         0.171
career_appearances_pre         1.000                   1.000             0.993               0.995                      0.944         0.176
career_goals_pre               0.993                   0.993             1.000               0.993                      0.929         0.155
career_assists_pre             0.993                   0.995             0.993               1.000                      0.959         0.179
career_goals_per_game_pre      0.935                   0.944             0.929               0.959                      1.000         0.222
target_goals                   0.171                   0.176             0.155               0.179                      0.222      

## Audit missigness in the working feature set

In [27]:
# Audit missing values in the working predictors

missing_summary = (
    df[engineered_feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

missing_df = pd.DataFrame({
    "missing_count": missing_summary,
    "missing_pct": (
        missing_summary / len(df) * 100
    ).round(2)
})

print("Features with missing values:")
print(missing_df.to_string())

print("\nNumber of affected features:", len(missing_df))
print("Total missing cells:", int(missing_summary.sum()))

print("\nRows with at least one missing predictor:")
rows_with_missing = (
    df[engineered_feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

print(rows_with_missing)

print("\nFirst occurrence of missingness by feature:")

for col in missing_df.index:
    first_idx = df[col].isna().idxmax()

    print(
        f"{col:<55} "
        f"{df.loc[first_idx, 'date'].date()} | "
        f"{df.loc[first_idx, 'team']} vs "
        f"{df.loc[first_idx, 'opponent']}"
    )

Features with missing values:
                                                 missing_count  missing_pct
opponent_clean_sheet_rate_h2h_scored_match_pre             313        23.48
opponent_goals_against_per_h2h_scored_match_pre            313        23.48
opponent_goals_for_per_h2h_scored_match_pre                313        23.48
team_goals_for_last_5_scored_matches_pre                     6         0.45
team_goals_against_last_5_scored_matches_pre                 6         0.45
days_since_last_goal                                         5         0.38
days_rest                                                    1         0.08

Number of affected features: 7
Total missing cells: 957

Rows with at least one missing predictor:
314

First occurrence of missingness by feature:
opponent_clean_sheet_rate_h2h_scored_match_pre          2002-08-14 | Sporting CP vs Inter Milan
opponent_goals_against_per_h2h_scored_match_pre         2002-08-14 | Sporting CP vs Inter Milan
opponent_goals_for_pe

In [29]:
# Verify that missingness is structural

h2h_missing_cols = [
    "opponent_goals_for_per_h2h_scored_match_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",
    "opponent_clean_sheet_rate_h2h_scored_match_pre",
]

print("H2H missingness vs score-data coverage:")

for col in h2h_missing_cols:
    missing = df[col].isna()

    print(
        f"{col:<55} "
        f"missing={missing.sum():>3} | "
        f"coverage=0 among missing="
        f"{(df.loc[missing, 'opponent_score_data_h2h_matches_pre'] == 0).sum():>3}"
    )


print("\nTeam recent-score missingness:")

for col in [
    "team_goals_for_last_5_scored_matches_pre",
    "team_goals_against_last_5_scored_matches_pre",
]:
    missing = df[col].isna()

    print(f"\n{col}")
    print(
        df.loc[
            missing,
            [
                "date",
                "team",
                "team_appearances_with_ronaldo_pre",
                "team_score_coverage_last_5_pre",
            ]
        ].to_string(index=False)
    )


print("\nMissing days_since_last_goal:")

print(
    df.loc[
        df["days_since_last_goal"].isna(),
        [
            "date",
            "career_appearances_pre",
            "career_goals_pre",
            "days_since_last_goal",
        ]
    ].to_string(index=False)
)


print("\nMissing days_rest:")

print(
    df.loc[
        df["days_rest"].isna(),
        [
            "date",
            "career_appearances_pre",
            "days_rest",
        ]
    ].to_string(index=False)
)

H2H missingness vs score-data coverage:
opponent_goals_for_per_h2h_scored_match_pre             missing=313 | coverage=0 among missing=313
opponent_goals_against_per_h2h_scored_match_pre         missing=313 | coverage=0 among missing=313
opponent_clean_sheet_rate_h2h_scored_match_pre          missing=313 | coverage=0 among missing=313

Team recent-score missingness:

team_goals_for_last_5_scored_matches_pre
      date              team  team_appearances_with_ronaldo_pre  team_score_coverage_last_5_pre
2002-08-14       Sporting CP                                  0                               0
2003-08-16 Manchester United                                  0                               0
2003-08-20          Portugal                                  0                               0
2009-08-29       Real Madrid                                  0                               0
2018-08-18          Juventus                                  0                               0
2023-01-22   

In [31]:
# Create structural missingness indicators

df["no_h2h_score_history_pre"] = (
    df["opponent_score_data_h2h_matches_pre"] == 0
).astype(int)

df["no_team_score_history_pre"] = (
    df["team_score_coverage_last_5_pre"] == 0
).astype(int)

df["no_prior_career_goal_pre"] = (
    df["career_goals_pre"] == 0
).astype(int)

df["first_career_appearance"] = (
    df["career_appearances_pre"] == 0
).astype(int)


structural_indicators = [
    "no_h2h_score_history_pre",
    "no_team_score_history_pre",
    "no_prior_career_goal_pre",
    "first_career_appearance",
]


print("Structural indicators:\n")

for col in structural_indicators:
    print(
        f"{col:<35} "
        f"1s={df[col].sum():>4} | "
        f"0s={(df[col] == 0).sum():>4}"
    )


print("\nConsistency checks:")

print(
    "H2H indicator matches H2H NaNs:",
    (
        df["no_h2h_score_history_pre"].eq(1)
        ==
        df["opponent_goals_for_per_h2h_scored_match_pre"].isna()
    ).all()
)

print(
    "Team-history indicator matches team-score NaNs:",
    (
        df["no_team_score_history_pre"].eq(1)
        ==
        df["team_goals_for_last_5_scored_matches_pre"].isna()
    ).all()
)

print(
    "No-prior-goal indicator matches days-since-goal NaNs:",
    (
        df["no_prior_career_goal_pre"].eq(1)
        ==
        df["days_since_last_goal"].isna()
    ).all()
)

print(
    "First-appearance indicator matches days-rest NaN:",
    (
        df["first_career_appearance"].eq(1)
        ==
        df["days_rest"].isna()
    ).all()
)

Structural indicators:

no_h2h_score_history_pre            1s= 313 | 0s=1020
no_team_score_history_pre           1s=   6 | 0s=1327
no_prior_career_goal_pre            1s=   5 | 0s=1328
first_career_appearance             1s=   1 | 0s=1332

Consistency checks:
H2H indicator matches H2H NaNs: True
Team-history indicator matches team-score NaNs: True
No-prior-goal indicator matches days-since-goal NaNs: True
First-appearance indicator matches days-rest NaN: True


In [33]:
# Fill structural missing values

structural_missing_features = [
    "opponent_goals_for_per_h2h_scored_match_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",
    "opponent_clean_sheet_rate_h2h_scored_match_pre",
    "team_goals_for_last_5_scored_matches_pre",
    "team_goals_against_last_5_scored_matches_pre",
    "days_since_last_goal",
    "days_rest",
]

# Keep a copy before filling so we can verify exactly what changed
missing_before = df[structural_missing_features].isna().sum()

df[structural_missing_features] = (
    df[structural_missing_features]
    .fillna(0)
)

missing_after = df[structural_missing_features].isna().sum()

print("Missing values before and after filling:\n")

missing_check = pd.DataFrame({
    "before": missing_before,
    "after": missing_after,
})

print(missing_check.to_string())

print("\nTotal missing values across all current predictors:")

current_features = (
    engineered_feature_columns
    + structural_indicators
)

print(
    df[current_features]
    .isna()
    .sum()
    .sum()
)

print("\nCurrent feature count:", len(current_features))

Missing values before and after filling:

                                                 before  after
opponent_goals_for_per_h2h_scored_match_pre         313      0
opponent_goals_against_per_h2h_scored_match_pre     313      0
opponent_clean_sheet_rate_h2h_scored_match_pre      313      0
team_goals_for_last_5_scored_matches_pre              6      0
team_goals_against_last_5_scored_matches_pre          6      0
days_since_last_goal                                  5      0
days_rest                                             1      0

Total missing values across all current predictors:
0

Current feature count: 80


In [35]:
# Group current features by football/modeling concept

feature_groups = {
    "career_state": [
        "age_years",
        "career_appearances_pre",
        "career_goals_pre",
        "career_assists_pre",
        "career_goals_per_game_pre",
    ],

    "recent_ronaldo_form": [
        col for col in current_features
        if (
            "last_" in col
            or "ewma" in col
            or "streak" in col
            or "since_last_goal" in col
        )
        and not col.startswith("team_")
        and "opponent" not in col
    ],

    "schedule_fatigue": [
        col for col in current_features
        if (
            "days_rest" in col
            or "matches_last_" in col
            or "congestion" in col
        )
    ],

    "team_context": [
        col for col in current_features
        if col.startswith("team_")
    ],

    "ronaldo_vs_opponent": [
        col for col in current_features
        if (
            "opponent_prior_meetings" in col
            or "ronaldo_goals_vs_opponent" in col
            or "ronaldo_assists_vs_opponent" in col
            or "ronaldo_goals_per_game_vs_opponent" in col
            or "ronaldo_scoring_rate_vs_opponent" in col
            or "ronaldo_goals_last_3_vs_opponent" in col
        )
    ],

    "team_vs_opponent_h2h": [
        col for col in current_features
        if (
            "h2h" in col
            and col not in [
                "opponent_prior_meetings_pre",
            ]
        )
    ],

    "elo_strength": [
        col for col in current_features
        if "elo" in col
    ],

    "season_calendar": [
        col for col in current_features
        if (
            "season_" in col
            or col in ["month", "day_of_week"]
            or "congestion" in col
        )
    ],

    "categorical_context": categorical_features,
}


for group, features in feature_groups.items():
    unique_features = list(dict.fromkeys(features))

    print(f"\n{'=' * 65}")
    print(f"{group.upper()} ({len(unique_features)})")
    print("=" * 65)

    for feature in unique_features:
        print(feature)


CAREER_STATE (5)
age_years
career_appearances_pre
career_goals_pre
career_assists_pre
career_goals_per_game_pre

RECENT_RONALDO_FORM (16)
matches_last_7d
matches_last_14d
matches_last_30d
goals_last_3
assists_last_5
assists_last_10
goals_per_game_last_5
goals_per_game_last_10
goals_per_game_last_20
scoring_rate_last_5
scoring_rate_last_10
multi_goal_rate_last_10
goals_ewma_pre
assists_ewma_pre
goalless_streak_games_pre
days_since_last_goal

SCHEDULE_FATIGUE (5)
days_rest
matches_last_7d
matches_last_14d
matches_last_30d
is_dec_jan_congestion

TEAM_CONTEXT (14)
team_appearances_with_ronaldo_pre
team_goals_by_ronaldo_pre
team_ronaldo_goals_per_game_pre
team_points_last_5_ronaldo_apps_pre
team_points_last_10_ronaldo_apps_pre
team_win_rate_last_5_ronaldo_apps_pre
team_unbeaten_rate_last_5_ronaldo_apps_pre
team_goals_for_last_5_scored_matches_pre
team_goals_against_last_5_scored_matches_pre
team_score_coverage_last_5_pre
team_h2h_wins_pre
team_h2h_draws_pre
team_h2h_losses_pre
team_matchup

In [37]:
# Check conceptual feature coverage

all_grouped_features = set()

for features in feature_groups.values():
    all_grouped_features.update(features)

ungrouped_features = [
    col for col in current_features
    if col not in all_grouped_features
]

print("Current model features:", len(current_features))
print("Unique features captured by groups:", len(all_grouped_features))

print("\nUngrouped features:", len(ungrouped_features))

for col in ungrouped_features:
    print(" -", col)

Current model features: 80
Unique features captured by groups: 68

Ungrouped features: 12
 - is_national_team
 - competition_importance
 - consecutive_scoring_games_pre
 - competition_appearances_pre
 - competition_goals_pre
 - competition_goals_per_game_pre
 - venue_appearances_pre
 - venue_goals_pre
 - venue_goals_per_game_pre
 - no_team_score_history_pre
 - no_prior_career_goal_pre
 - first_career_appearance


## Understand opponent Elo coverage

In [40]:
# Inspect opponent-strength coverage

first_time_opponent = df["opponent_prior_meetings_pre"] == 0

print("First-time opponents:", first_time_opponent.sum())

print("\nOpponent Elo for first-time opponents:")
print(
    df.loc[
        first_time_opponent,
        [
            "opponent_matchup_elo_pre",
            "opponent_matchup_elo_games_pre",
            "matchup_elo_diff_pre",
        ]
    ]
    .describe()
    .round(3)
    .to_string()
)

print("\nOpponent Elo game-count distribution for first-time opponents:")
print(
    df.loc[
        first_time_opponent,
        "opponent_matchup_elo_games_pre"
    ]
    .value_counts()
    .sort_index()
    .head(20)
    .to_string()
)

print("\nFirst 15 first-time opponents:")

print(
    df.loc[
        first_time_opponent,
        [
            "date",
            "team",
            "opponent",
            "opponent_prior_meetings_pre",
            "opponent_matchup_elo_pre",
            "opponent_matchup_elo_games_pre",
            "matchup_elo_diff_pre",
        ]
    ]
    .head(15)
    .to_string(index=False)
)

First-time opponents: 283

Opponent Elo for first-time opponents:
       opponent_matchup_elo_pre  opponent_matchup_elo_games_pre  matchup_elo_diff_pre
count                   283.000                         283.000               283.000
mean                   1500.836                           1.141               118.278
std                      13.489                          17.449                76.532
min                    1500.000                           0.000               -22.180
25%                    1500.000                           0.000                49.285
50%                    1500.000                           0.000               121.330
75%                    1500.000                           0.000               173.835
max                    1726.730                         292.000               294.780

Opponent Elo game-count distribution for first-time opponents:
opponent_matchup_elo_games_pre
0      281
31       1
292      1

First 15 first-time opponents:


In [44]:
# Engineer Elo reliability features

df["opponent_elo_unseen_pre"] = (
    df["opponent_matchup_elo_games_pre"] == 0
).astype(int)

df["opponent_elo_experience_log_pre"] = np.log1p(
    df["opponent_matchup_elo_games_pre"]
)

elo_engineered_features = [
    "opponent_elo_unseen_pre",
    "opponent_elo_experience_log_pre",
]

current_features.extend(elo_engineered_features)


print("New Elo features:\n")

for col in elo_engineered_features:
    print(
        f"{col:<40} "
        f"min={df[col].min():.3f} | "
        f"median={df[col].median():.3f} | "
        f"max={df[col].max():.3f}"
    )


print("\nUnseen/default Elo rows:")
print(df["opponent_elo_unseen_pre"].sum())

print(
    "All unseen rows have Elo 1500:",
    (
        df.loc[
            df["opponent_elo_unseen_pre"] == 1,
            "opponent_matchup_elo_pre"
        ] == 1500
    ).all()
)

print(
    "All unseen rows have zero Elo games:",
    (
        df.loc[
            df["opponent_elo_unseen_pre"] == 1,
            "opponent_matchup_elo_games_pre"
        ] == 0
    ).all()
)

print("\nCurrent feature count:", len(current_features))
print(
    "Duplicate feature names:",
    len(current_features) - len(set(current_features))
)

New Elo features:

opponent_elo_unseen_pre                  min=0.000 | median=0.000 | max=1.000
opponent_elo_experience_log_pre          min=0.000 | median=1.386 | max=5.694

Unseen/default Elo rows:
281
All unseen rows have Elo 1500: True
All unseen rows have zero Elo games: True

Current feature count: 82
Duplicate feature names: 0


In [46]:
# Inspect opponent-history coverage

def opponent_history_group(n):
    if n == 0:
        return "0 — unseen"
    elif n <= 2:
        return "1–2 meetings"
    elif n <= 5:
        return "3–5 meetings"
    elif n <= 10:
        return "6–10 meetings"
    else:
        return "11+ meetings"


df["opponent_history_group"] = (
    df["opponent_prior_meetings_pre"]
    .apply(opponent_history_group)
)

history_summary = (
    df.groupby(
        "opponent_history_group",
        observed=True
    )
    .agg(
        matches=("match_id", "count"),
        goals=("target_goals", "sum"),
        scoring_matches=("target_scored", "sum"),
        avg_prior_meetings=("opponent_prior_meetings_pre", "mean"),
    )
)

history_summary["pct_dataset"] = (
    history_summary["matches"] / len(df) * 100
)

history_summary["goals_per_game"] = (
    history_summary["goals"] / history_summary["matches"]
)

history_summary["scoring_rate"] = (
    history_summary["scoring_matches"] / history_summary["matches"]
)

order = [
    "0 — unseen",
    "1–2 meetings",
    "3–5 meetings",
    "6–10 meetings",
    "11+ meetings",
]

history_summary = history_summary.reindex(order)

print(history_summary.round(3).to_string())

print("\nUnique opponents:", df["opponent"].nunique())

print(
    "Opponents appearing fewer than 5 times:",
    (df["opponent"].value_counts() < 5).sum()
)

print(
    "Opponents appearing only once:",
    (df["opponent"].value_counts() == 1).sum()
)

                        matches  goals  scoring_matches  avg_prior_meetings  pct_dataset  goals_per_game  scoring_rate
opponent_history_group                                                                                                
0 — unseen                  283    171              123               0.000       21.230           0.604         0.435
1–2 meetings                350    242              158               1.409       26.257           0.691         0.451
3–5 meetings                302    258              167               3.884       22.656           0.854         0.553
6–10 meetings               239    188              126               7.649       17.929           0.787         0.527
11+ meetings                159    119               83              16.384       11.928           0.748         0.522

Unique opponents: 283
Opponents appearing fewer than 5 times: 184
Opponents appearing only once: 76


In [48]:
# Engineer opponent familiarity

df["first_time_opponent_pre"] = (
    df["opponent_prior_meetings_pre"] == 0
).astype(int)

df["opponent_meetings_log_pre"] = np.log1p(
    df["opponent_prior_meetings_pre"]
)

opponent_engineered_features = [
    "first_time_opponent_pre",
    "opponent_meetings_log_pre",
]

current_features.extend(opponent_engineered_features)

print("Opponent familiarity features:\n")

for col in opponent_engineered_features:
    print(
        f"{col:<35} "
        f"min={df[col].min():.3f} | "
        f"median={df[col].median():.3f} | "
        f"max={df[col].max():.3f}"
    )

print("\nFirst-time opponent rows:")
print(df["first_time_opponent_pre"].sum())

print(
    "Indicator exactly matches zero prior meetings:",
    (
        df["first_time_opponent_pre"].eq(1)
        ==
        df["opponent_prior_meetings_pre"].eq(0)
    ).all()
)

print("\nCurrent feature count:", len(current_features))

print(
    "Duplicate feature names:",
    len(current_features) - len(set(current_features))
)

print(
    "Remaining missing values:",
    df[current_features].isna().sum().sum()
)

Opponent familiarity features:

first_time_opponent_pre             min=0.000 | median=0.000 | max=1.000
opponent_meetings_log_pre           min=0.000 | median=1.386 | max=3.611

First-time opponent rows:
283
Indicator exactly matches zero prior meetings: True

Current feature count: 84
Duplicate feature names: 0
Remaining missing values: 0


In [50]:
# Inspect complete candidate feature set

print("Total current features:", len(current_features))

for i, col in enumerate(current_features, start=1):
    print(f"{i:>2}. {col}")

Total current features: 84
 1. season
 2. team
 3. opponent
 4. competition
 5. competition_type
 6. venue
 7. is_national_team
 8. competition_importance
 9. age_years
10. career_appearances_pre
11. career_goals_pre
12. career_assists_pre
13. career_goals_per_game_pre
14. days_rest
15. matches_last_7d
16. matches_last_14d
17. matches_last_30d
18. goals_last_3
19. assists_last_5
20. assists_last_10
21. goals_per_game_last_5
22. goals_per_game_last_10
23. goals_per_game_last_20
24. scoring_rate_last_5
25. scoring_rate_last_10
26. multi_goal_rate_last_10
27. goals_ewma_pre
28. assists_ewma_pre
29. consecutive_scoring_games_pre
30. goalless_streak_games_pre
31. days_since_last_goal
32. season_appearances_pre
33. season_goals_pre
34. season_assists_pre
35. season_goals_per_game_pre
36. team_appearances_with_ronaldo_pre
37. team_goals_by_ronaldo_pre
38. team_ronaldo_goals_per_game_pre
39. team_points_last_5_ronaldo_apps_pre
40. team_points_last_10_ronaldo_apps_pre
41. team_win_rate_last_5_r

## Classify feature by forecast-time availability

In [54]:
# Classify features by forecast-time availability

fixture_known_features = [
    "season",
    "team",
    "opponent",
    "competition",
    "competition_type",
    "venue",
    "is_national_team",
    "competition_importance",
    "age_years",
    "days_rest",
    "matches_last_7d",
    "matches_last_14d",
    "matches_last_30d",
    "month",
    "day_of_week",
    "season_progress_pct",
    "is_season_opening_phase",
    "is_season_run_in",
    "is_dec_jan_congestion",
]

stateful_features = [
    col for col in current_features
    if col not in fixture_known_features
]

print("Fixture-known features:", len(fixture_known_features))
print("Stateful/reconstructable features:", len(stateful_features))
print(
    "Total classified:",
    len(fixture_known_features) + len(stateful_features)
)

print("\nUnclassified:")
print([
    col for col in current_features
    if col not in fixture_known_features
    and col not in stateful_features
])

print("\nDuplicate classifications:")
print(
    set(fixture_known_features)
    .intersection(stateful_features)
)

print("\nSTATEFUL FEATURES")
print("-" * 60)

for i, col in enumerate(stateful_features, 1):
    print(f"{i:>2}. {col}")

Fixture-known features: 19
Stateful/reconstructable features: 65
Total classified: 84

Unclassified:
[]

Duplicate classifications:
set()

STATEFUL FEATURES
------------------------------------------------------------
 1. career_appearances_pre
 2. career_goals_pre
 3. career_assists_pre
 4. career_goals_per_game_pre
 5. goals_last_3
 6. assists_last_5
 7. assists_last_10
 8. goals_per_game_last_5
 9. goals_per_game_last_10
10. goals_per_game_last_20
11. scoring_rate_last_5
12. scoring_rate_last_10
13. multi_goal_rate_last_10
14. goals_ewma_pre
15. assists_ewma_pre
16. consecutive_scoring_games_pre
17. goalless_streak_games_pre
18. days_since_last_goal
19. season_appearances_pre
20. season_goals_pre
21. season_assists_pre
22. season_goals_per_game_pre
23. team_appearances_with_ronaldo_pre
24. team_goals_by_ronaldo_pre
25. team_ronaldo_goals_per_game_pre
26. team_points_last_5_ronaldo_apps_pre
27. team_points_last_10_ronaldo_apps_pre
28. team_win_rate_last_5_ronaldo_apps_pre
29. team_un

## Check unseen-category risk with a temporal split

In [57]:
# Diagnose unseen categorical values over time

split_idx = int(len(df) * 0.80)

train_diag = df.iloc[:split_idx]
test_diag = df.iloc[split_idx:]

print("Diagnostic split:")
print(
    train_diag["date"].min().date(),
    "to",
    train_diag["date"].max().date()
)
print(
    test_diag["date"].min().date(),
    "to",
    test_diag["date"].max().date()
)

print("\nUnseen categories in later 20%:\n")

for col in categorical_features:
    train_values = set(train_diag[col].dropna().unique())
    test_values = set(test_diag[col].dropna().unique())

    unseen = sorted(test_values - train_values)

    unseen_rows = test_diag[col].isin(unseen).sum()

    print(f"{col:<20}")
    print(f"  Train unique: {len(train_values)}")
    print(f"  Test unique:  {len(test_values)}")
    print(f"  Unseen values: {len(unseen)}")
    print(f"  Test rows affected: {unseen_rows}")

    if unseen:
        print(f"  Values: {unseen}")

    print()

Diagnostic split:
2002-08-14 to 2021-05-15
2021-05-19 to 2026-08-28

Unseen categories in later 20%:

season              
  Train unique: 19
  Test unique:  7
  Unseen values: 6
  Test rows affected: 260
  Values: ['2021/22', '2022/23', '2023/24', '2024/25', '2025/26', '2026/27']

team                
  Train unique: 5
  Test unique:  4
  Unseen values: 1
  Test rows affected: 151
  Values: ['Al-Nassr']

opponent            
  Train unique: 228
  Test unique:  105
  Unseen values: 55
  Test rows affected: 172
  Values: ['Abha', 'Al Ahli Doha', 'Al Ain', 'Al Baten', 'Al Khaleej Saihat', 'Al Kholood', 'Al Najma', 'Al Okhdood', 'Al Orubah', 'Al Riyadh', 'Al Shabab', 'Al Shorta', 'Al Taawon', 'Al Taee', 'Al Wehda Club', "Al Zawra'a", 'Al-Adalah', 'Al-Ahli Jeddah', 'Al-Duhail SC', 'Al-Ettifaq', 'Al-Fateh', 'Al-Fayha', 'Al-Gharafa', 'Al-Hazm', 'Al-Hilal', 'Al-Ittihad FC', 'Al-Qadisiyah FC', 'Al-Raed', 'Al-Rayyan SC', 'Al-Wasl FC', 'Brentford', 'Brighton', 'Burnley', 'Colombia', 'Congo DR', 

In [59]:
# Separate stable and identity-based categorical features

stable_categorical_features = [
    "competition_type",
    "venue",
    "day_of_week",
]

identity_categorical_features = [
    "season",
    "team",
    "opponent",
    "competition",
]

print("Stable categorical features:")
for col in stable_categorical_features:
    print(
        f" - {col:<20} "
        f"unique={df[col].nunique()}"
    )

print("\nIdentity/regime categorical features:")
for col in identity_categorical_features:
    print(
        f" - {col:<20} "
        f"unique={df[col].nunique()}"
    )


# Context features that can help the model generalize
# when a literal identity is unseen.
generalization_features = [
    "is_national_team",
    "competition_importance",
    "age_years",

    "season_appearances_pre",
    "season_goals_per_game_pre",

    "team_ronaldo_goals_per_game_pre",
    "team_points_last_5_ronaldo_apps_pre",
    "team_points_last_10_ronaldo_apps_pre",
    "team_goals_for_last_5_scored_matches_pre",
    "team_goals_against_last_5_scored_matches_pre",

    "opponent_prior_meetings_pre",
    "ronaldo_goals_per_game_vs_opponent_pre",
    "ronaldo_scoring_rate_vs_opponent_pre",

    "opponent_h2h_points_per_game_pre",
    "opponent_h2h_unbeaten_rate_pre",

    "team_matchup_elo_pre",
    "opponent_matchup_elo_pre",
    "matchup_elo_diff_pre",

    "competition_goals_per_game_pre",
    "venue_goals_per_game_pre",

    "first_time_opponent_pre",
    "opponent_elo_unseen_pre",
]

print("\nGeneralization/context features available:",
      len(generalization_features))

print(
    "All present:",
    all(col in current_features for col in generalization_features)
)

Stable categorical features:
 - competition_type     unique=4
 - venue                unique=3
 - day_of_week          unique=7

Identity/regime categorical features:
 - season               unique=25
 - team                 unique=6
 - opponent             unique=283
 - competition          unique=31

Generalization/context features available: 22
All present: True


In [ ]:
# Step 3W — Check remaining numeric redundancy

numeric_current_features = [
    col for col in current_features
    if pd.api.types.is_numeric_dtype(df[col])
]

corr_matrix = df[numeric_current_features].corr()

high_corr_pairs = []

for i in range(len(numeric_current_features)):
    for j in range(i + 1, len(numeric_current_features)):
        col_a = numeric_current_features[i]
        col_b = numeric_current_features[j]

        corr = corr_matrix.loc[col_a, col_b]

        if abs(corr) >= 0.95:
            high_corr_pairs.append(
                (col_a, col_b, corr)
            )

high_corr_pairs = sorted(
    high_corr_pairs,
    key=lambda x: abs(x[2]),
    reverse=True
)

print("Numeric current features:", len(numeric_current_features))
print("Pairs with |correlation| >= 0.95:", len(high_corr_pairs))
print()

for col_a, col_b, corr in high_corr_pairs:
    print(
        f"{col_a:<40} <-> "
        f"{col_b:<40} "
        f"{corr:.6f}"
    )

In [61]:
# Check remaining numeric redundancy

numeric_current_features = [
    col for col in current_features
    if pd.api.types.is_numeric_dtype(df[col])
]

corr_matrix = df[numeric_current_features].corr()

high_corr_pairs = []

for i in range(len(numeric_current_features)):
    for j in range(i + 1, len(numeric_current_features)):
        col_a = numeric_current_features[i]
        col_b = numeric_current_features[j]

        corr = corr_matrix.loc[col_a, col_b]

        if abs(corr) >= 0.95:
            high_corr_pairs.append(
                (col_a, col_b, corr)
            )

high_corr_pairs = sorted(
    high_corr_pairs,
    key=lambda x: abs(x[2]),
    reverse=True
)

print("Numeric current features:", len(numeric_current_features))
print("Pairs with |correlation| >= 0.95:", len(high_corr_pairs))
print()

for col_a, col_b, corr in high_corr_pairs:
    print(
        f"{col_a:<40} <-> "
        f"{col_b:<40} "
        f"{corr:.6f}"
    )

Numeric current features: 77
Pairs with |correlation| >= 0.95: 14

age_years                                <-> career_appearances_pre                   0.999524
opponent_elo_unseen_pre                  <-> first_time_opponent_pre                  0.995513
career_appearances_pre                   <-> career_assists_pre                       0.995126
age_years                                <-> career_goals_pre                         0.993404
career_appearances_pre                   <-> career_goals_pre                         0.993392
career_goals_pre                         <-> career_assists_pre                       0.993174
age_years                                <-> career_assists_pre                       0.992575
venue_appearances_pre                    <-> venue_goals_pre                          0.981242
career_appearances_pre                   <-> venue_appearances_pre                    0.968986
age_years                                <-> venue_appearances_pre            

In [63]:
# Inspect disagreement between opponent novelty indicators

novelty_disagreement = df[
    df["first_time_opponent_pre"]
    != df["opponent_elo_unseen_pre"]
][
    [
        "date",
        "team",
        "opponent",
        "competition",
        "opponent_prior_meetings_pre",
        "opponent_matchup_elo_pre",
        "opponent_matchup_elo_games_pre",
        "first_time_opponent_pre",
        "opponent_elo_unseen_pre",
        "target_goals",
    ]
]

print("Total matches:", len(df))

print(
    "First-time opponent rows:",
    df["first_time_opponent_pre"].sum()
)

print(
    "Elo-unseen rows:",
    df["opponent_elo_unseen_pre"].sum()
)

print(
    "Indicator disagreement rows:",
    len(novelty_disagreement)
)

print("\nDisagreement cases:\n")

print(
    novelty_disagreement.to_string(index=False)
)

Total matches: 1333
First-time opponent rows: 283
Elo-unseen rows: 281
Indicator disagreement rows: 2

Disagreement cases:

      date              team          opponent           competition  opponent_prior_meetings_pre  opponent_matchup_elo_pre  opponent_matchup_elo_games_pre  first_time_opponent_pre  opponent_elo_unseen_pre  target_goals
2007-09-19 Manchester United       Sporting CP UEFA Champions League                            0                   1509.95                              31                        1                        0             1
2013-02-13       Real Madrid Manchester United UEFA Champions League                            0                   1726.73                             292                        1                        0             1


In [65]:
# Mechanical target-leakage sanity check

numeric_current_features = [
    col for col in current_features
    if pd.api.types.is_numeric_dtype(df[col])
]

target_corr_rows = []

for feature in numeric_current_features:
    for target in [
        "target_goals",
        "target_scored",
        "target_multi_goal",
    ]:
        corr = df[feature].corr(df[target])

        target_corr_rows.append(
            {
                "feature": feature,
                "target": target,
                "correlation": corr,
                "abs_correlation": abs(corr),
            }
        )

target_corr_df = (
    pd.DataFrame(target_corr_rows)
    .sort_values(
        "abs_correlation",
        ascending=False
    )
)

print("Numeric predictors checked:", len(numeric_current_features))

print("\nTop 20 predictor-target correlations:")
print(
    target_corr_df[
        ["feature", "target", "correlation"]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nSuspicious |correlation| >= 0.80:")
suspicious = target_corr_df[
    target_corr_df["abs_correlation"] >= 0.80
]

if suspicious.empty:
    print("None")
else:
    print(
        suspicious[
            ["feature", "target", "correlation"]
        ].to_string(index=False)
    )

print("\nForbidden post-match columns in current_features:")
print(
    sorted(
        set(current_features)
        .intersection(postmatch_reference_columns)
    )
)

print("\nTarget columns in current_features:")
print(
    sorted(
        set(current_features)
        .intersection(target_columns)
    )
)

Numeric predictors checked: 77

Top 20 predictor-target correlations:
                        feature        target  correlation
team_ronaldo_goals_per_game_pre target_scored     0.284317
team_ronaldo_goals_per_game_pre  target_goals     0.282585
         goals_per_game_last_20 target_scored     0.254223
 competition_goals_per_game_pre  target_goals     0.243625
      career_goals_per_game_pre target_scored     0.243618
      season_goals_per_game_pre target_scored     0.243489
 competition_goals_per_game_pre target_scored     0.239801
       venue_goals_per_game_pre target_scored     0.235332
           matchup_elo_diff_pre  target_goals     0.235043
         goals_per_game_last_20  target_goals     0.231393
           scoring_rate_last_10 target_scored     0.229708
       venue_goals_per_game_pre  target_goals     0.226252
           matchup_elo_diff_pre target_scored     0.225081
           team_matchup_elo_pre target_scored     0.224631
      season_goals_per_game_pre  target_goals

In [67]:
# Freeze Phase 3 feature specification

FULL_FEATURES = current_features.copy()

STABLE_CATEGORICALS = [
    "competition_type",
    "venue",
    "day_of_week",
]

IDENTITY_CATEGORICALS = [
    "season",
    "team",
    "opponent",
    "competition",
]

CATEGORICAL_FEATURES = (
    STABLE_CATEGORICALS
    + IDENTITY_CATEGORICALS
)

NUMERIC_FEATURES = [
    col for col in FULL_FEATURES
    if col not in CATEGORICAL_FEATURES
]

TARGET = "target_goals"

SECONDARY_TARGETS = [
    "target_scored",
    "target_multi_goal",
]

# Final integrity checks
assert len(FULL_FEATURES) == len(set(FULL_FEATURES))
assert set(CATEGORICAL_FEATURES).issubset(FULL_FEATURES)
assert set(NUMERIC_FEATURES).issubset(FULL_FEATURES)

assert set(NUMERIC_FEATURES).isdisjoint(
    CATEGORICAL_FEATURES
)

assert len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES) == len(FULL_FEATURES)

assert TARGET not in FULL_FEATURES
assert not set(SECONDARY_TARGETS).intersection(FULL_FEATURES)
assert not set(postmatch_reference_columns).intersection(FULL_FEATURES)

assert df[FULL_FEATURES].isna().sum().sum() == 0


print("PHASE 3 FEATURE SPECIFICATION")
print("=" * 50)

print("Full candidate features:", len(FULL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

print("\nStable categoricals:")
print(STABLE_CATEGORICALS)

print("\nIdentity/regime categoricals:")
print(IDENTITY_CATEGORICALS)

print("\nPrimary target:")
print(TARGET)

print("\nSecondary targets:")
print(SECONDARY_TARGETS)

print("\nMissing predictor values:")
print(df[FULL_FEATURES].isna().sum().sum())

print("\nDuplicate feature names:")
print(len(FULL_FEATURES) - len(set(FULL_FEATURES)))

print("\nForbidden columns present:")
print(
    sorted(
        set(FULL_FEATURES)
        .intersection(
            target_columns
            + postmatch_reference_columns
            + identifier_columns
        )
    )
)

PHASE 3 FEATURE SPECIFICATION
Full candidate features: 84
Numeric features: 77
Categorical features: 7

Stable categoricals:
['competition_type', 'venue', 'day_of_week']

Identity/regime categoricals:
['season', 'team', 'opponent', 'competition']

Primary target:
target_goals

Secondary targets:
['target_scored', 'target_multi_goal']

Missing predictor values:
0

Duplicate feature names:
0

Forbidden columns present:
[]


In [69]:
# Save final engineered dataset

from pathlib import Path

OUTPUT_DIR = Path("../data/engineered")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "ronaldo_1000_ml_engineered.csv"

# Remove analysis-only helper column before saving
if "opponent_history_group" in df.columns:
    df = df.drop(columns=["opponent_history_group"])

df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nCareer goals:")
print(df["target_goals"].sum())

print("\nUnique match IDs:")
print(df["match_id"].nunique())

print("\nDate range:")
print(
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

print("\nEngineered features present:")
for col in [
    "no_h2h_score_history_pre",
    "no_team_score_history_pre",
    "no_prior_career_goal_pre",
    "first_career_appearance",
    "opponent_elo_unseen_pre",
    "opponent_elo_experience_log_pre",
    "first_time_opponent_pre",
    "opponent_meetings_log_pre",
]:
    print(f"{col:<40} {col in df.columns}")

print("\nPredictor missing values:")
print(df[FULL_FEATURES].isna().sum().sum())

Saved: ..\data\engineered\ronaldo_1000_ml_engineered.csv
Shape: (1333, 100)
Rows: 1333
Columns: 100

Career goals:
978

Unique match IDs:
1333

Date range:
2002-08-14 to 2026-08-28

Engineered features present:
no_h2h_score_history_pre                 True
no_team_score_history_pre                True
no_prior_career_goal_pre                 True
first_career_appearance                  True
opponent_elo_unseen_pre                  True
opponent_elo_experience_log_pre          True
first_time_opponent_pre                  True
opponent_meetings_log_pre                True

Predictor missing values:
0


In [71]:
# Step 3AB — Save feature specification

import json
from pathlib import Path

ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

feature_spec = {
    "full_features": FULL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "stable_categoricals": STABLE_CATEGORICALS,
    "identity_categoricals": IDENTITY_CATEGORICALS,
    "primary_target": TARGET,
    "secondary_targets": SECONDARY_TARGETS,
    "identifier_columns": identifier_columns,
    "postmatch_reference_columns": postmatch_reference_columns,
    "deterministic_exclusions": deterministic_exclusions,
}

FEATURE_SPEC_PATH = ARTIFACT_DIR / "feature_spec.json"

with open(FEATURE_SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(
        feature_spec,
        f,
        indent=2
    )

print("Saved:", FEATURE_SPEC_PATH)

print("\nFeature specification summary:")
print("Full features:", len(feature_spec["full_features"]))
print("Numeric features:", len(feature_spec["numeric_features"]))
print("Categorical features:", len(feature_spec["categorical_features"]))
print("Primary target:", feature_spec["primary_target"])
print("Secondary targets:", feature_spec["secondary_targets"])

Saved: ..\artifacts\feature_spec.json

Feature specification summary:
Full features: 84
Numeric features: 77
Categorical features: 7
Primary target: target_goals
Secondary targets: ['target_scored', 'target_multi_goal']
